# 실험 3. 조합 변수

- 기준: `exp01`의 기준 모델 (LightGBM 5-Fold, OOF AUC 0.95866)
- 가설: 비행거리가 짧으면 같은 지연이라도 불만이 다를 수 있다 → 거리 대비 지연
- 방법: 파생 변수를 추가하고 같은 Fold, 같은 파라미터로 OOF AUC 비교

## 환경설정

In [1]:
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
START_TIME = time.time()

## 데이터 경로 탐색

In [2]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)

# 크로스플랫폼 경로 자동 해결

def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))  # drop duplicates, keep order

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.\n"
            "- Kaggle: 오른쪽 Input 패널에서 대회 데이터를 추가하세요.\n"
            "- 로컬: 노트북과 같은 위치의 data/ 폴더에 파일을 넣으세요."
        )
    raise RuntimeError(
        "데이터 폴더 후보가 여러 개입니다. 사용할 폴더를 DATA_DIR에 직접 지정하세요:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

DATA_DIR = /Users/youngjin/ai/kaggle_practice/Kaggle_Airline_Satisfaction/data


## 데이터 로드 및 전처리

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "satisfaction"
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]

In [4]:
y = train[TARGET].astype(int)
X = train.drop(columns=["id", TARGET]).copy()
X_test = test.drop(columns=["id"]).copy()

for col in CAT_COLS:
    categories = sorted(train[col].unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)

assert list(X.columns) == list(X_test.columns)
print("X      :", X.shape)
print("X_test :", X_test.shape)
print("y mean :", round(y.mean(), 4))

X      : (699635, 21)
X_test : (299844, 21)
y mean : 0.4436


## 교차검증 함수

In [5]:
LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "seed": SEED,
    "verbose": -1,
}
N_FOLDS = 5


def run_cv(X, X_test, params=LGB_PARAMS):
    """Stratified 5-fold LightGBM; returns the OOF AUC and the averaged test prediction."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_pred = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
        dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

        oof_pred[va_idx] = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
        test_pred += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS
        print(f"Fold {fold}: AUC = {roc_auc_score(y.iloc[va_idx], oof_pred[va_idx]):.5f}   trees = {model.best_iteration}")

    return roc_auc_score(y, oof_pred), test_pred

In [6]:
results = {}

## 기준 모델 (원본 변수)

In [7]:
auc, _ = run_cv(X, X_test)
results["기준 (원본 변수)"] = auc
print(f"\nOOF AUC = {auc:.5f}")

Fold 1: AUC = 0.95877   trees = 549


Fold 2: AUC = 0.95759   trees = 293


Fold 3: AUC = 0.95930   trees = 286


Fold 4: AUC = 0.95870   trees = 280


Fold 5: AUC = 0.95902   trees = 256

OOF AUC = 0.95866


## 파생 변수 생성

In [8]:
CLASS_TRAVEL = sorted((train["Class"] + "_" + train["Type of Travel"]).unique())


def interaction_features(df):
    class_travel = df["Class"].astype(str) + "_" + df["Type of Travel"].astype(str)
    dep = df["Departure Delay in Minutes"]
    arr = df["Arrival Delay in Minutes"]
    return pd.DataFrame({
        "class_travel": pd.Categorical(class_travel, categories=CLASS_TRAVEL),
        "delay_per_100km": dep / (df["Flight Distance"] / 100),
        "total_delay": dep + arr.fillna(0),
    })


F = interaction_features(X)
F_test = interaction_features(X_test)
F.describe(include="all").round(2)

,class_travel,delay_per_100km,total_delay
count,699635,699635.00,699635.00
unique,6,NaN,NaN
top,Business_Business travel,NaN,NaN
freq,338272,NaN,NaN
mean,NaN,0.14,2.31
std,NaN,0.88,11.25
min,NaN,0.00,0.00
25%,NaN,0.00,0.00
50%,NaN,0.00,0.00
75%,NaN,0.00,0.00


## 실험 3-1: class_travel 추가 (등급 × 여행 목적)

- 이유: 등급과 여행 목적을 조합하면 만족 비율이 0.097~0.732로 크게 갈림 (Business석 개인 여행은 11%)

In [9]:
FEATS = ["class_travel"]
auc, _ = run_cv(pd.concat([X, F[FEATS]], axis=1), pd.concat([X_test, F_test[FEATS]], axis=1))
results["class_travel 추가"] = auc
print(f"\nOOF AUC = {auc:.5f}   (기준 대비 {auc - results['기준 (원본 변수)']:+.5f})")

Fold 1: AUC = 0.95885   trees = 375


Fold 2: AUC = 0.95762   trees = 327


Fold 3: AUC = 0.95936   trees = 351


Fold 4: AUC = 0.95863   trees = 228


Fold 5: AUC = 0.95907   trees = 258

OOF AUC = 0.95869   (기준 대비 +0.00003)


## 실험 3-2: delay_per_100km 추가 (거리 대비 출발 지연)

- 이유: 짧은 비행의 지연은 덜 불만일 것이라는 가설. 거리 대비 지연의 타깃 상관 -0.054로 출발 지연(-0.036)보다 약간 높음

In [10]:
FEATS = ["delay_per_100km"]
auc, _ = run_cv(pd.concat([X, F[FEATS]], axis=1), pd.concat([X_test, F_test[FEATS]], axis=1))
results["delay_per_100km 추가"] = auc
print(f"\nOOF AUC = {auc:.5f}   (기준 대비 {auc - results['기준 (원본 변수)']:+.5f})")

Fold 1: AUC = 0.95882   trees = 697


Fold 2: AUC = 0.95750   trees = 198


Fold 3: AUC = 0.95929   trees = 368


Fold 4: AUC = 0.95868   trees = 253


Fold 5: AUC = 0.95896   trees = 288

OOF AUC = 0.95865   (기준 대비 -0.00001)
